In [ ]:
# TemporalBench Adversarial Suite — Reversion, CausalReasoning, Interval
# 8 adversarial task families: Reversion, Interval, CausalReasoning, MultiReversion,
# IntervalMidpoint, MultiEntityJoin, FutureFact, TimelineReconstruction

import json
import numpy as np
from typing import Dict, List, Optional

In [ ]:
class Fact:
    def __init__(self, key: str, value: str, valid_from: int, valid_to: int, accesses: int = 0):
        self.key = key
        self.value = value
        self.valid_from = valid_from
        self.valid_to = valid_to
        self.accesses = accesses

    def is_valid_at(self, day: int) -> bool:
        return self.valid_from <= day < self.valid_to

    def age(self, day: int) -> float:
        return day - self.valid_from

    def decay_score(self, day: int, half_life: float = 50.0) -> float:
        return np.exp(-0.693 * self.age(day) / half_life)

    def attention_score(self) -> float:
        return 1.0 + 0.001 * np.log1p(self.accesses)


class TemporalAttentionStoreWithValidity:
    """System D_revised — validity windows as hard gate. Unaffected by adversarial conditions."""
    def __init__(self, half_life: float = 50.0):
        self.half_life = half_life
        self.facts: Dict[str, List[Fact]] = {}


    def put(self, key, value, valid_from, valid_to, accesses=0):
        if key not in self.facts:
            self.facts[key] = []
        self.facts[key].append(Fact(key, value, valid_from, valid_to, accesses))

    def get(self, key, as_of_day=None):
        if key not in self.facts:
            return None
        candidates = [f for f in self.facts[key] if f.is_valid_at(as_of_day)]
        if not candidates:
            return None
        scores = [(f.decay_score(as_of_day, self.half_life) * f.attention_score(), f) for f in candidates]
        return max(scores, key=lambda x: x[0])[1].value

    def get_valid_interval(self, key: str, as_of_day: int) -> Optional[tuple]:
        if key not in self.facts:
            return None
        for f in self.facts[key]:
            if f.is_valid_at(as_of_day):
                return (f.valid_from, f.valid_to)
        return None

    def get_history(self, key: str) -> List[Fact]:
        if key not in self.facts:
            return []
        return sorted(self.facts[key], key=lambda f: f.valid_from)

In [ ]:
def load_adversarial(path: str) -> List[Dict]:
    with open(path + '.jsonl') as f:
        return [json.loads(line) for line in f]



def score_adversarial(question: Dict, store: TemporalAttentionStoreWithValidity) -> float:
    task_family = question.get('task_family', 'Interval')
    query_day = question.get('query_day') or question.get('day')
    key = question.get('key', '')
    expected = question.get('expected_answer') or question.get('answer') or ''

    if task_family == 'Reversion':
        history = store.get_history(key)
        for i, f in enumerate(history):
            if f.is_valid_at(query_day):
                if i > 0:
                    prev = history[i - 1]
                    return 1.0 if prev.value == expected else 0.0
                return 0.0
        return 0.0

    elif task_family == 'Interval':
        iv = store.get_valid_interval(key, query_day)
        if expected.lower() in ('yes', 'true', 'valid', '1'):
            return 1.0 if iv is not None else 0.0
        else:
            return 0.0 if iv is not None else 1.0

    elif task_family == 'CausalReasoning':
        predicted = store.get(key, query_day)
        if predicted == expected:
            return 1.0
        if predicted and expected:
            pred_tokens = set(str(predicted).lower().split())
            exp_tokens = set(str(expected).lower().split())
            return len(pred_tokens & exp_tokens) / max(len(exp_tokens), 1)
        return 0.0

    elif task_family == 'MultiReversion':
        history = store.get_history(key)
        predicted_vals = [f.value for f in history if f.is_valid_at(query_day)]
        expected_list = expected.split('|') if '|' in expected else [expected]
        if len(predicted_vals) >= len(expected_list):
            return 1.0 if predicted_vals[-len(expected_list):] == expected_list else 0.0
        return 0.0

    elif task_family == 'IntervalMidpoint':
        iv = store.get_valid_interval(key, query_day)
        if iv is None:
            return 0.0
        midpoint = (iv[0] + iv[1]) // 2
        predicted = store.get(key, midpoint)
        return 1.0 if predicted == expected else 0.0

    elif task_family == 'MultiEntityJoin':
        key2 = question.get('key2', '')
        val1 = store.get(key, query_day)
        val2 = store.get(key2, query_day)
        expected_parts = expected.split('|')
        if len(expected_parts) == 2:
            return 1.0 if val1 == expected_parts[0] and val2 == expected_parts[1] else 0.0
        return 0.0

    elif task_family == 'FutureFact':
        iv = store.get_valid_interval(key, query_day)
        if iv is not None:
            return 0.0
        return 1.0 if expected.lower() in ('none', 'not yet', 'unknown', 'n/a') else 0.0

    elif task_family == 'TimelineReconstruction':
        history = store.get_history(key)
        predicted = ' -> '.join([f"{f.valid_from}:{f.value}" for f in history])
        pred_tokens = set(str(predicted).lower().split())
        exp_tokens = set(str(expected).lower().split())
        return len(pred_tokens & exp_tokens) / max(len(exp_tokens), 1)

    else:
        predicted = store.get(key, query_day)
        return 1.0 if predicted == expected else 0.0

In [ ]:
def run_adversarial_evaluation(questions: List[Dict], store_class) -> Dict:
    store = store_class()
    results = {}

    for q in questions:
        if 'facts' in q:
            for fd in q['facts']:
                store.put(fd['key'], fd['value'], fd['valid_from'], fd['valid_to'],
                          fd.get('accesses', 0))

        tf = q.get('task_family', 'Unknown')
        if tf not in results:
            results[tf] = {'correct': 0.0, 'total': 0}

        score = score_adversarial(q, store)
        results[tf]['correct'] += score
        results[tf]['total'] += 1

    metrics = {tf: r['correct'] / r['total'] if r['total'] > 0 else 0.0 for tf, r in results.items()}
    trs = np.mean(list(metrics.values())) if metrics else 0.0
    return {'metrics': metrics, 'trs': trs}

In [ ]:
DATA_PATH = '/kaggle/input/datasets/zacharymaronek/temporalbenchmark/temporalbench/data/benchmarks/adversarial_temporal_questions'

print('Loading TemporalBench Adversarial Suite...')
questions = load_adversarial(DATA_PATH)
print(f'Loaded {len(questions)} adversarial questions')

print('\nEvaluating D_revised (Validity Windows)...')
results = run_adversarial_evaluation(questions, TemporalAttentionStoreWithValidity)
print(f'TRS: {results["trs"]:.4f}')
for tf, acc in results['metrics'].items():
    print(f'  {tf:30s}: {acc:.1%}')


print('\n--- NOTE ---')
print('D_revised with validity windows should score near 100% on Interval, Reversion,')
print('MultiReversion, IntervalMidpoint, and FutureFact task families.')